# Diffusion Models

我们离开 Score Matching，来到传统意义上的 Diffusion Model。不过后续我们会读到 https://arxiv.org/abs/2011.13456 Score-Based Generative Modeling through Stochastic Differential Equations，这篇文章揭示了 Score Matching 与本章所说模型的本质共通之处。

我们先说说 Diffusion Model 共通的背景。我认为要先讲讲 Variational Autoencoder (自变分编码器)。不过 VAE 应该只是简单一提，因为实在太早期了，我们只是了解动机。

# Variational Autoencoder

VAE 的出发点仍是拟合真实数据分布 $p_{data}$。VAE 认为，假设我们的数据 $\mathbf{x}$ 是由某种隐变量 $\mathbf{z}$（例如：猫的品种、姿态、颜色）生成的。我们想要最大化似然 $p(\mathbf{x})$：$$p(x) = \int p(x, z) dz = \int p(x|z)p(z) dz$$
问题是 $p(z|x) = \frac{p(x, z)}{p(x)}$ 难以计算。我们不知道对于特定图像究竟对应了潜空间中的哪个部分。

我们引入一个相对简单的、参数化的分布 $q_\phi(z|x)$（称为变分分布），去尝试逼近那个复杂的真实后验 $p(z|x)$

做变换$$\ln p(x) = \ln \int p(x, z) dz$$
构造期望形式$$\ln p(x) = \ln \int q_\phi(z|x) \frac{p(x, z)}{q_\phi(z|x)} dz = \ln \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \frac{p(x, z)}{q_\phi(z|x)} \right]$$
由 Jensen 不等式 $\ln \mathbb{E}[X] \ge \mathbb{E}[\ln X]$，得到$$\ln p(x) \ge \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \ln \frac{p(x, z)}{q_\phi(z|x)} \right] \equiv \text{ELBO}$$
右式被称为证据下界 Evidence Lower Bound

更进一步的，实际上存在这样的关系$$\ln p(x) = \text{ELBO} + D_{KL}(q_\phi(z|x) \parallel p(z|x))$$

其中 $D_{KL}(q_\phi(z|x) \parallel p(z|x))$ 是 KL 离散度。如果你不知道什么是 KL 离散度，简单来说就是衡量两个分布差异的量，始终大于等于 0。具体是$$D_{KL}(p_{\text{data}} \parallel p_\theta) = \int p_{\text{data}}(x) \log \frac{p_{\text{data}}(x)}{p_\theta(x)} dx$$ 在此处具体是$$D_{KL}(q_\phi(z|x) \parallel p(z|x)) = \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \ln \frac{q_\phi(z|x)}{p(z|x)} \right]$$

我们证明一下刚刚提到的关系。根据贝叶斯公式 $p(z|x) = \frac{p(x, z)}{p(x)}$ $$D_{KL}(q_\phi(z|x) \parallel p(z|x)) = \mathbb{E}_{q_\phi} \left[ \ln \frac{q_\phi(z|x)}{p(x, z) / p(x)} \right]$$对数展开 $$D_{KL}(q_\phi(z|x) \parallel p(z|x)) = \mathbb{E}_{q_\phi} \left[ \ln \left( \frac{q_\phi(z|x)}{p(x, z)} \cdot p(x) \right) \right]$$ $$= \mathbb{E}_{q_\phi} \left[ \ln \frac{q_\phi(z|x)}{p(x, z)} + \ln p(x) \right]$$得到 $$D_{KL}(q_\phi(z|x) \parallel p(z|x)) = \mathbb{E}_{q_\phi} \left[ \ln \frac{q_\phi(z|x)}{p(x, z)} \right] + \mathbb{E}_{q_\phi} \left[ \ln p(x) \right]$$将与分布无关的 $\ln p(x)$ 写为常量$$D_{KL}(q_\phi(z|x) \parallel p(z|x)) = \mathbb{E}_{q_\phi} \left[ \ln \frac{q_\phi(z|x)}{p(x, z)} \right] + \ln p(x)$$

根据 ELBO 定义，有$$\text{ELBO} = \mathbb{E}_{q_\phi(z|x)} \left[ \ln \frac{p(x, z)}{q_\phi(z|x)} \right]$$因此$$D_{KL}(q_\phi(z|x) \parallel p(z|x)) = -\text{ELBO} + \ln p(x)$$移项得证$$\ln p(x) = \text{ELBO} + D_{KL}(q_\phi(z|x) \parallel p(z|x))$$

所以现在很一目了然。如果我们想要优化 KL 离散度 (使得模型学习的分布尽量接近真实数据)，实际上就是在推高 ELBO 的值。
总而言之，VAE 就是学习 $p(z|x)$，所以给 VAE 一张真实图片 $x$，VAE 擅长找到图片对应的隐变量 $z$，在真实推理时 Decoder 还原即可。

一个关于 ELBO 的额外结论是 $$\text{ELBO} = \underbrace{\mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})} [\ln p_\theta(\mathbf{x}|\mathbf{z})]}_{\text{Reconstruction Loss}} - \underbrace{D_{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))}_{\text{Regularization (KL)}}$$其中重建项确保 Decoder 能根据隐变量还原出原图，正则项确保每个 $\mathbf{x}$ 映射出的 $\mathbf{z}$ 都尽量服从标准正态分布。

更本质的，VAE 实际上在学习如何将真实数据分布映射到预设的潜空间，这个潜空间我们一般直接设为高斯空间。因此 VAE 的推理也是从纯高斯噪声开始的。实际上高斯噪声就是高斯分布，模型知道真实空间如何映射到高斯分布空间，也知道如何映射回去。

我们再解释得详细一些。VAE 实际上将最开始的纯噪声和最终的图片分割到了两个空间里，这很反直觉。潜空间 $\mathcal{Z}$ 是一个低维、抽象、规则的数学空间（通常是 $\mathbb{R}^d$）。在这里，每一个坐标点 $z$ 就代表真实空间中的某种图片。数据空间 $\mathcal{X}$ 是高维、具体、杂乱的像素空间（$\mathbb{R}^D$，$D \gg d$）。当采样一个高斯噪声时，实际上是在 $\mathcal{Z}$ 空间选了一个点。模型会将这个点翻译到真实的数据空间图像。

我们关于 VAE 就说到这里，这不是学习的重点，而是理解现代 Diffusion 的雏形与动机的切入点。VAE 只有两层 $\mathbf{x}$ 和 $\mathbf{z}$。它试图一步完成压缩和解压。我们随后介绍的 Denoising Diffusion Probabilistic Models 则可以看作多层的 VAE，我们一步一步还原出原图。我们单开一节详细说说。

# Denoising Diffusion Probabilistic Models

DDPM 以及后来者，以及之前提到的 Score Matching，与 VAE 有着非常大的不同。不同就在 VAE 将初始的高斯噪声设定为潜空间存在，最终的生成数据设定为另外一个真实数据空间的存在，并且学习一个空间之间的映射。然而，DDPM 或 SM 却将潜空间与真实数据空间统一，将初始的高斯噪声视为真实数据空间中的一张脏图片并且试图多步还原到原本的干净图片。

不建议在这个节点就开始接触 VAE，SM 与 DDPM 的大一统，我觉得反而会更不助于理解。你可以暂时将 VAE 视为一个异类，但是 VAE 确实是后来众多架构的鼻祖。现在我们开始说 DDPM。

### 目标函数与基本逻辑

推荐你读 https://arxiv.org/abs/2006.11239 Denoising Diffusion Probabilistic Models 这是 DDPM 的原论文。DDPM 的核心思想绝大程度来自于热力学。简单来说，我们将纯高斯噪声视为一张原本真实干净图片通过多步加噪获得，所以我们希望模型学习这个多步加噪的逆向，即如何通过多步减噪获得原始的干净图片。

定义前向逻辑，称为扩散过程，我们研究一张干净图片如何一步一步变成纯噪声。本质是一个 Markovian 链加噪$$q(\mathbf{x}_{1:T} | \mathbf{x}_0) := \prod_{t=1}^T q(\mathbf{x}_t | \mathbf{x}_{t-1})$$
其中$$q(\mathbf{x}_t | \mathbf{x}_{t-1}) := \mathcal{N}(\mathbf{x}_t; \sqrt{1-\beta_t}\mathbf{x}_{t-1}, \beta_t \mathbf{I})$$
$\beta_t$ 是一个允许被调度的系数，对于每个时间步 $t$ 都会分配。一般会按照线性赋值。另外的，你会注意到$\sqrt{1-\beta_t}$这个系数是为了方差归一化专门安排的。

定义逆向逻辑，本质是减噪。这部分需要网络去学习$$p_\theta(\mathbf{x}_{0:T}) := p(\mathbf{x}_T) \prod_{t=1}^T p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t)$$

一个神奇的结论，来自 Sohl-Dickstein 等人，他们证明在 $\beta_t$ 选取很小时，逆向逻辑的每一步实际上也近似一个高斯分布。如果你感兴趣，可以看到$$p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t) := \mathcal{N}(\mathbf{x}_{t-1}; \boldsymbol{\mu}_\theta(\mathbf{x}_t, t), \boldsymbol{\Sigma}_\theta(\mathbf{x}_t, t))$$我们后续会详细谈这个结论。

关于目标函数，我们希望最小化$-\ln p_\theta(\mathbf{x}_0)$，这里和 Score Matching 是共通的。直接的想法是计算 $p_\theta(\mathbf{x}_0) = \int p_\theta(\mathbf{x}_{0:T}) d\mathbf{x}_{1:T}$，但是实际上不可行，因为对于模型内部函数积分是极度复杂的。

我们做一个变换$$-\ln p_\theta(\mathbf{x}_0) = -\ln \int q(\mathbf{x}_{1:T} | \mathbf{x}_0) \frac{p_\theta(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} d\mathbf{x}_{1:T}$$
依旧根据 Jensen 不等式 $-\ln \mathbb{E}[X] \le \mathbb{E}[-\ln X]$，有$$-\ln p_\theta(\mathbf{x}_0) \le \mathbb{E}_q \left[ -\ln \frac{p_\theta(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} \right] =: L$$
进一步化简$$-\ln \frac{p_\theta(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} = -\ln \left( \frac{p(\mathbf{x}_T) \prod_{t=1}^T p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t)}{\prod_{t=1}^T q(\mathbf{x}_t | \mathbf{x}_{t-1})} \right)$$
最终得到$$L = \mathbb{E}_q \left[ -\ln p(\mathbf{x}_T) - \sum_{t=1}^T \ln \frac{p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t)}{q(\mathbf{x}_t | \mathbf{x}_{t-1})} \right]$$

无论你是否注意到，实际上 $\mathbb{E}_q \left[ -\ln \frac{p_\theta(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} \right]$ 非常像 VAE 的 ELBO 结果 $\mathbb{E}_{q_\phi(z|x)} \left[ \ln \frac{p(x, z)}{q_\phi(z|x)} \right]$。如果你将整个 DDPM 的目标函数展开，这一步甚至会更加一目了然。我为你演示

根据贝叶斯公式$$q(\mathbf{x}_t | \mathbf{x}_{t-1}) = \frac{q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0) q(\mathbf{x}_t | \mathbf{x}_0)}{q(\mathbf{x}_{t-1} | \mathbf{x}_0)}$$
代入得到$$\ln \frac{p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t)}{q(\mathbf{x}_t | \mathbf{x}_{t-1})} = \ln \frac{p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t) q(\mathbf{x}_{t-1} | \mathbf{x}_0)}{q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0) q(\mathbf{x}_t | \mathbf{x}_0)}$$
拆分$$\ln \frac{p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t)}{q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0)} + \ln \frac{q(\mathbf{x}_{t-1} | \mathbf{x}_0)}{q(\mathbf{x}_t | \mathbf{x}_0)}$$

当你将上式带入最原始 $L$，会发现有一部分项完全被抵消$$\sum_{t=1}^T \left( \ln q(\mathbf{x}_{t-1} | \mathbf{x}_0) - \ln q(\mathbf{x}_t | \mathbf{x}_0) \right) = \ln q(\mathbf{x}_0 | \mathbf{x}_0) - \ln q(\mathbf{x}_T | \mathbf{x}_0)$$
重新组合 $L$，得到$$L = \mathbb{E}_q [ \underbrace{D_{KL}(q(\mathbf{x}_T | \mathbf{x}_0) \| p(\mathbf{x}_T))}_{L_T} + \sum_{t=2}^T \underbrace{D_{KL}(q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0) \| p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t))}_{L_{t-1}} \underbrace{- \ln p_\theta(\mathbf{x}_0 | \mathbf{x}_1)}_{L_0} ]$$

为你解释每一项含义。 

$L_T$ 是先验匹配项 (Prior Matching)，在 VAE 对应 $D_{KL}(q(z|x) \| p(z))$。这一项实际上衡量最后一步得到的噪声是否真的服从我们预设的标准高斯先验。由于 $q$ 是固定的加噪过程，这一项在训练时几乎是常数。

$L_{t-1}$ 是一致性项 (Consistency/Denoising)，在 VAE 对应隐变量之间的转换约束。这是多步层级的体现。它要求模型预测的去噪步必须尽可能贴合知道 $x_0$ 的真实后验去噪步。这是模型学习的核心。

$L_0$ 是重建项 (Reconstruction)，在 VAE 对应 $\mathbb{E}_{q(z|x)}[\ln p(x|z)]$。这是最后一步从微弱噪声还原到像素的损失。

所以为什么说 DDPM 是多步 VAE ？在 VAE 中，$z$ 是隐变量。在 DDPM 中，每一个中间状态 $\mathbf{x}_1, \dots, \mathbf{x}_T$ 都可以被视为一层隐变量。前向过程 $q(\mathbf{x}_t | \mathbf{x}_{t-1})$ 是一个固定参数的编码器，而逆向过程 $p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t)$ 是一个可学习的解码器。上述推导出的 $L$ 正是 $T$ 层隐变量叠加后的变分下界（ELBO）。

所以DDPM 实际上是一个极其深层的，具有固定编码器，且每一层隐变量维度都与原数据一致的层级 VAE。

关于 DDPM 的基本逻辑，先说到这里。接下来我想要谈论一些数学变换，他们非常关键，这包括我刚刚所提到的逆向逻辑实际上是个高斯分布这个结论。

但是你要保留一个疑问，就是如何具体计算 $L$ ？我们现在实际上对目标函数还没有正式参数化。我预计会在讲完几个重要技巧之后为你演示 $L$ 的化简与具体计算。

### DDPM 的捷径与技巧

DDPM 的捷径大概有两条。第一条是，作者提出前向加噪中实际上可以从 $x_0$ 一步加噪到任意时间步的 $x_t$，因为高斯噪声的叠加还是高斯噪声。第二条是，逆向分布 $q(\mathbf{x}_{t-1} | \mathbf{x}_t)$ 的函数形式与前向过程是一致的，也是一个高斯分布。我们先说第一条。

我们证明第一条是正确的。令 $\alpha_t = 1 - \beta_t$，根据前向过程的定义，每一步加噪 $q(\mathbf{x}_t | \mathbf{x}_{t-1})$ 是：$$\mathbf{x}_t = \sqrt{\alpha_t} \mathbf{x}_{t-1} + \sqrt{1 - \alpha_t} \boldsymbol{\epsilon}_{t-1}$$其中 $\boldsymbol{\epsilon}_{t-1} \sim \mathcal{N}(0, \mathbf{I})$ 是该步骤加入的噪声。

我们将 $\mathbf{x}_{t-1}$ 的表达式代入 $\mathbf{x}_t$：$$\mathbf{x}_{t-1} = \sqrt{\alpha_{t-1}} \mathbf{x}_{t-2} + \sqrt{1 - \alpha_{t-1}} \boldsymbol{\epsilon}_{t-2}$$代入后得到：$$\mathbf{x}_t = \sqrt{\alpha_t} (\sqrt{\alpha_{t-1}} \mathbf{x}_{t-2} + \sqrt{1 - \alpha_{t-1}} \boldsymbol{\epsilon}_{t-2}) + \sqrt{1 - \alpha_t} \boldsymbol{\epsilon}_{t-1}$$ $$\mathbf{x}_t = \sqrt{\alpha_t \alpha_{t-1}} \mathbf{x}_{t-2} + (\sqrt{\alpha_t (1 - \alpha_{t-1})} \boldsymbol{\epsilon}_{t-2} + \sqrt{1 - \alpha_t} \boldsymbol{\epsilon}_{t-1})$$

注意到 $\boldsymbol{\epsilon}_{t-1}$ 和 $\boldsymbol{\epsilon}_{t-2}$ 是两个相互独立的标准正态分布。根据高斯分布的性质：两个独立正态分布相加，其结果仍为正态分布，且方差等于各自方差之和。

总方差为：$(\sqrt{\alpha_t (1 - \alpha_{t-1})})^2 + (\sqrt{1 - \alpha_t})^2 = 1 - \alpha_t \alpha_{t-1}$

因此，我们可以用一个新的标准正态噪声 $\bar{\boldsymbol{\epsilon}} \sim \mathcal{N}(0, \mathbf{I})$ 来替换这两个噪声的组合：$$\mathbf{x}_t = \sqrt{\alpha_t \alpha_{t-1}} \mathbf{x}_{t-2} + \sqrt{1 - \alpha_t \alpha_{t-1}} \bar{\boldsymbol{\epsilon}}$$

以此类推，如果我们一直迭代回 $\mathbf{x}_0$，系数会不断累乘。我们定义 $\bar{\alpha}_t = \prod_{i=1}^t \alpha_i$，那么最终公式就变成了：$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t} \boldsymbol{\epsilon}$$其中 $\boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})$。

第一条捷径解决了训练准备数据集的并行性，我们无需一步一步加噪，而是可以直达我们想要的特定时间步 $t$。

我们来看第二条捷径。我们已经提到了这个结论，逆向减噪实际上也是一个高斯分布。我们正式证明这一点。我们要证明的是 $q(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0)$ 是高斯分布。

根据贝叶斯定理：$$q(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0) = \frac{q(\mathbf{x}_t|\mathbf{x}_{t-1}, \mathbf{x}_0) q(\mathbf{x}_{t-1}|\mathbf{x}_0)}{q(\mathbf{x}_t|\mathbf{x}_0)}$$

由于 Markovian 性质，第一项 $q(\mathbf{x}_t|\mathbf{x}_{t-1}, \mathbf{x}_0) = q(\mathbf{x}_t|\mathbf{x}_{t-1})$。更具体的，我们忽略系数来看

$q(\mathbf{x}_t|\mathbf{x}_{t-1})$: $\exp \left( -\frac{(\mathbf{x}_t - \sqrt{\alpha_t}\mathbf{x}_{t-1})^2}{2\beta_t} \right)$ 

$q(\mathbf{x}_{t-1}|\mathbf{x}_0)$: $\exp \left( -\frac{(\mathbf{x}_{t-1} - \sqrt{\bar{\alpha}_{t-1}}\mathbf{x}_0)^2}{2(1-\bar{\alpha}_{t-1})} \right)$ 

$q(\mathbf{x}_t|\mathbf{x}_0)$: $\exp \left( -\frac{(\mathbf{x}_t - \sqrt{\bar{\alpha}_t}\mathbf{x}_0)^2}{2(1-\bar{\alpha}_t)} \right)$

得到$$\text{Exponent Part} \propto -\frac{1}{2} \left[ \frac{(\mathbf{x}_t - \sqrt{\alpha_t}\mathbf{x}_{t-1})^2}{\beta_t} + \frac{(\mathbf{x}_{t-1} - \sqrt{\bar{\alpha}_{t-1}}\mathbf{x}_0)^2}{1-\bar{\alpha}_{t-1}} - \frac{(\mathbf{x}_t - \sqrt{\bar{\alpha}_t}\mathbf{x}_0)^2}{1-\bar{\alpha}_t} \right]$$

我们只关注 $\mathbf{x}_{t-1}$ 的项：$$\text{Exp} = -\frac{1}{2} \left[ \underbrace{\frac{(\mathbf{x}_t - \sqrt{\alpha_t}\mathbf{x}_{t-1})^2}{\beta_t}}_{来自 q(\mathbf{x}_t|\mathbf{x}_{t-1})} + \underbrace{\frac{(\mathbf{x}_{t-1} - \sqrt{\bar{\alpha}_{t-1}}\mathbf{x}_0)^2}{1-\bar{\alpha}_{t-1}}}_{来自 q(\mathbf{x}_{t-1}|\mathbf{x}_0)} \right] + C(\mathbf{x}_t, \mathbf{x}_0)$$
我们将括号展开，整理成关于 $\mathbf{x}_{t-1}$ 的标准二次型 $A\mathbf{x}_{t-1}^2 - 2B\mathbf{x}_{t-1} + C$：
$$A = \frac{\alpha_t}{\beta_t} + \frac{1}{1-\bar{\alpha}_{t-1}} = \frac{\alpha_t(1-\bar{\alpha}_{t-1}) + \beta_t}{\beta_t(1-\bar{\alpha}_{t-1})} = \frac{\alpha_t - \bar{\alpha}_t + 1 - \alpha_t}{\beta_t(1-\bar{\alpha}_{t-1})} = \frac{1-\bar{\alpha}_t}{\beta_t(1-\bar{\alpha}_{t-1})}$$
$$B = \frac{\sqrt{\alpha_t}\mathbf{x}_t}{\beta_t} + \frac{\sqrt{\bar{\alpha}_{t-1}}\mathbf{x}_0}{1-\bar{\alpha}_{t-1}}$$


在一个正态分布 $\exp(-\frac{1}{2\sigma^2}(x-\mu)^2)$ 中，系数对应关系为：$\frac{1}{\sigma^2} = A$，$\frac{\mu}{\sigma^2} = B$。

方差 $\tilde{\beta}_t$：$$\tilde{\beta}_t = \frac{1}{A} = \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t} \cdot \beta_t$$
均值 $\tilde{\boldsymbol{\mu}}_t$：$$\tilde{\boldsymbol{\mu}}_t = \frac{B}{A} = \left( \frac{\sqrt{\alpha_t}}{\beta_t}\mathbf{x}_t + \frac{\sqrt{\bar{\alpha}_{t-1}}}{1-\bar{\alpha}_{t-1}}\mathbf{x}_0 \right) \cdot \frac{\beta_t(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t}$$化简得到
$$\tilde{\boldsymbol{\mu}}_t = \frac{\sqrt{\alpha_t}(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t} \mathbf{x}_t + \frac{\sqrt{\bar{\alpha}_{t-1}}\beta_t}{1-\bar{\alpha}_t} \mathbf{x}_0$$

问题来了，实际推理时我们并不知道 $x_0$，但是我们可以通过第一条捷径中的方法反推$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\boldsymbol{\epsilon} \implies \mathbf{x}_0 = \frac{1}{\sqrt{\bar{\alpha}_t}}(\mathbf{x}_t - \sqrt{1-\bar{\alpha}_t}\boldsymbol{\epsilon})$$
把这个 $\mathbf{x}_0$ 代入上面的 $\tilde{\boldsymbol{\mu}}_t$ 公式：$$\tilde{\boldsymbol{\mu}}_t = \frac{\sqrt{\alpha_t}(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t} \mathbf{x}_t + \frac{\sqrt{\bar{\alpha}_{t-1}}\beta_t}{1-\bar{\alpha}_t} \left[ \frac{1}{\sqrt{\bar{\alpha}_t}}(\mathbf{x}_t - \sqrt{1-\bar{\alpha}_t}\boldsymbol{\epsilon}) \right]$$

整合一下$$\tilde{\boldsymbol{\mu}}_t = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \boldsymbol{\epsilon} \right)$$

第二条捷径的结果告诉我们，去噪后的均值其实就是当前的脏图减去它所包含的噪声。网络预测出噪声，就等价于还原了均值。

这里有一点需要强调：在训练时，逆向过程确实是完全的高斯分布，但是真实推理时，由于我们并不知道实际的 $x_0$，这仅仅是一个近似高斯分布的过程。原因是对于一个连续时间的扩散过程，如果前向步长 $dt$ 趋向于 0，那么逆向过程的转移概率与前向过程具有相同的函数形式。由于我们选取的 $\beta_t$ 非常小（通常在 $10^{-4}$ 量级），每一时间步的变换极小。在这个微元极限下，即使真实分布 $p(\mathbf{x})$ 极其复杂，其局部逆向变换依然可以被高斯分布完美近似。

接下来为你讲述目标函数 $L$ 具体的化简与计算。

### 目标函数的化简

我们讨论一下刚刚所说的目标函数在实际训练中的化简。别忘记我们已有的结果$$L = \mathbb{E}_q [ \underbrace{D_{KL}(q(\mathbf{x}_T | \mathbf{x}_0) \| p(\mathbf{x}_T))}_{L_T} + \sum_{t=2}^T \underbrace{D_{KL}(q(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_0) \| p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t))}_{L_{t-1}} \underbrace{- \ln p_\theta(\mathbf{x}_0 | \mathbf{x}_1)}_{L_0} ]$$

$L_T$ 这一项衡量的是 $q(\mathbf{x}_T | \mathbf{x}_0)$ 与先验分布 $p(\mathbf{x}_T)$ 之间的距离。我们直接将前向过程的方差 $\beta_t$ 固定为常数（不做参数化学习），这就导致整个 $L_T$ 实际上都是预设的。我们在训练中忽略他。

再来看 $L_{t-1}$，我们讨论如何参数化逆向分布 $p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t) = \mathcal{N}(\mathbf{x}_{t-1}; \boldsymbol{\mu}_\theta, \boldsymbol{\Sigma}_\theta)$。原论文作者通过实验发现，学习这个方差并不容易，于是决定将其设为未经训练的随时间变化的常数 $\sigma_t^2 \mathbf{I}$。换言之 $$q(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_{t-1}; \tilde{\boldsymbol{\mu}}_t, \sigma_t^2 \mathbf{I})$$ $$p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t) = \mathcal{N}(\mathbf{x}_{t-1}; \boldsymbol{\mu}_\theta, \sigma_t^2 \mathbf{I})$$

根据 KL 散度的公式，对于两个 $d$ 维的高斯分布 $\mathcal{N}(\mu_1, \Sigma_1)$ 和 $\mathcal{N}(\mu_2, \Sigma_2)$：$$D_{KL} = \frac{1}{2} \left[ \text{tr}(\Sigma_2^{-1}\Sigma_1) + (\mu_2 - \mu_1)^T \Sigma_2^{-1} (\mu_2 - \mu_1) - d + \ln \frac{|\Sigma_2|}{|\Sigma_1|} \right]$$

当 $\Sigma_1 = \Sigma_2 = \sigma_t^2 \mathbf{I}$ 时：

$\text{tr}(\Sigma_2^{-1}\Sigma_1) = \text{tr}(\mathbf{I}) = d$ 

$\ln \frac{|\Sigma_2|}{|\Sigma_1|} = \ln 1 = 0$

中间那一项变为：$\frac{1}{\sigma_t^2} (\boldsymbol{\mu}_\theta - \tilde{\boldsymbol{\mu}}_t)^T (\boldsymbol{\mu}_\theta - \tilde{\boldsymbol{\mu}}_t) = \frac{1}{\sigma_t^2} \| \tilde{\boldsymbol{\mu}}_t - \boldsymbol{\mu}_\theta \|^2$

最终得到 $$L_{t-1} = \frac{1}{2\sigma_t^2} \| \tilde{\boldsymbol{\mu}}_t - \boldsymbol{\mu}_\theta \|^2$$
实际采样则是$$L_{t-1} = \mathbb{E}_q \left[ \frac{1}{2\sigma_t^2} \| \tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \mathbf{x}_0) - \boldsymbol{\mu}_\theta(\mathbf{x}_t, t) \|^2 \right] + C$$

所以现在的 $L_{t-1}$ 我们可以计算 $\tilde{\boldsymbol{\mu}}_t$ 的部分，来自我们刚刚说的捷径$$\tilde{\boldsymbol{\mu}}_t = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \boldsymbol{\epsilon} \right)$$

问题是如何计算 $\boldsymbol{\mu}_\theta$ ？原论文作者提出了一个参数化技巧（Reparameterization）。我们让神经网络输出一个噪声预测值 $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$，然后套入相同的结构
$$\boldsymbol{\mu}_\theta(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) \right) $$

现在我们把得到的结果全部代入 $L$ $$\left\| \left[ \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \boldsymbol{\epsilon} \right) \right] - \left[ \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \boldsymbol{\epsilon}_\theta \right) \right] \right\|^2$$

核心项变成了 $\| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) \|^2$。

我们证明了，最小化均值之间的距离，在数学上完全等价于最小化预测噪声与真实噪声之间的距离。

更多的$$L_{simple} = \sum_{t=1}^T \mathbb{E}_{\mathbf{x}_0 \sim q(\mathbf{x}_0), \boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})} \left[ \| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\underbrace{\sqrt{\bar{\alpha}_t}\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\boldsymbol{\epsilon}}_{\mathbf{x}_t \text{ 的边缘采样}}, t) \|^2 \right]$$

所以你可以看到，我们现在可以计算目标函数 $L$，而且实际上非常轻松。这里埋一个伏笔，观察到这个函数仅仅与边缘分布 $q(x_t|x_0)$ 相关，这是未来的优化切入点，不过我们现在可以暂时不理会。也许你还没反应过来。我为你展示完整的训练和推理流程。

### 训练与推理

#### 训练

在训练开始前，我们需要先确定噪声的分布调度。设定总步数 $T$（通常为 1000）。设定方差序列 $\beta_1, \dots, \beta_T$（通常是从 $10^{-4}$ 到 $0.02$ 的线性或余弦增长）。根据 $\beta_t$ 预计算出所有的 $\alpha_t = 1 - \beta_t$ 以及累乘系数 $\bar{\alpha}_t = \prod_{i=1}^t \alpha_i$。这些系数在整个训练过程中是固定不变的。

正式开始训练。每一个 Batch 的训练逻辑是，从训练集中随机抽取一个 Batch 的干净图片 $\mathbf{x}_0 \sim q(\mathbf{x}_0)$。

为这个 Batch 里的每一张图，随机从 $\{1, \dots, T\}$ 中抽一个整数 $t$。这样模型在一个 Batch 里就能同时看到轻微磨损的图和完全噪声的图，从而强迫网络学习所有尺度下的去噪能力。

采样噪声，生成一个与原图维度完全一致的标准高斯噪声：$\boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})$。

利用我们推导过的第一条捷径，不需要一步步加噪，直接加噪到第 $t$ 步的脏图状态：$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t} \boldsymbol{\epsilon}$$此时，$\mathbf{x}_t$ 就是喂给模型的脏图，而 $\boldsymbol{\epsilon}$ 就是标准答案。

将脏图 $\mathbf{x}_t$ 和对应的时间步 $t$ 给模型前向传播（通常是带 Time Embedding 的 U-Net）$$\hat{\boldsymbol{\epsilon}} = \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$$

直接计算预测噪声与真实噪声之间的 MSE 损失：$$L_{simple} = \| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) \|^2$$
在一个 Batch 结束之后，累加 $L$ 取平均逼近期望，再反向传播更新参数。

#### 推理

在 $t=T$ 时刻，我们不需要任何输入，直接从标准高斯分布中采样一个纯噪声：$$\mathbf{x}_T \sim \mathcal{N}(0, \mathbf{I})$$此时的 $\mathbf{x}_T$ 没有任何语义，只是数据空间里的一个随机点。

##### 我们需要运行一个从 $t=T$ 递减到 $1$ 的循环。在每一个步长 $t$ 中，执行以下操作

将当前的脏图 $\mathbf{x}_t$ 和步长 $t$ 给模型进行前向传播：$$\hat{\boldsymbol{\epsilon}} = \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$$模型输出它认为此时图像中包含的噪声成分。

计算去噪后的均值 $\boldsymbol{\mu}_\theta$ $$\boldsymbol{\mu}_\theta = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \hat{\boldsymbol{\epsilon}} \right)$$这一步是模型在尝试把当前的点往它认为的真实流形方向拉扯。

如果 $t > 1$，我们采样一个新的随机噪声 $\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$，并更新状态 $$\mathbf{x}_{t-1} = \boldsymbol{\mu}_\theta + \sigma_t \mathbf{z}$$ $\sigma_t$ 通常取 $\sqrt{\beta_t}$ 或 $\sqrt{\tilde{\beta}_t}$。我们上文中交代过取 $\sqrt{\beta_t}$。

这一步是我们在 SM 中熟悉的，Annaled Langevin Dynamics，加点噪声保证多样性。

当循环走到最后一步 $t=1$ 时，我们不再添加随机噪声 $\mathbf{z}$（即令 $\mathbf{z}=0$）。推理结束了。

更多的，我们展开刚刚的核心推理公式 $$\mathbf{x}_{t-1} = \underbrace{\frac{1}{\sqrt{\alpha_t}} \mathbf{x}_t}_{\text{缩放当前态}} - \underbrace{\frac{1 - \alpha_t}{\sqrt{\alpha_t}\sqrt{1 - \bar{\alpha}_t}} \boldsymbol{\epsilon}_\theta}_{\text{剥离预测的干扰}} + \underbrace{\sigma_t \mathbf{z}}_{\text{随机探索}}$$

这就是 DDPM 的推理全部过程。

### DDPM 的工程细节

原论文作者最后交代了几件事。我们假设的图片是连续的 $\mathbb{R}^n$，但实际的图片像素是离散的 $\{0, 1, \dots, 255\}$。为了处理这个问题，我们首先将将像素值从 $[0, 255]$ 线性缩放到了 $[-1, 1]$，在推理结束后放缩回去。其次的，在计算变分下界的最后一项 $L_0$（即从 $\mathbf{x}_1$ 生成 $\mathbf{x}_0$）时，我们使用一个近似来解决连续与离散的问题。如果你还没明白问题在哪，实际上就是高斯分布是一个连续分布，它在某个孤立点上的概率密度是没意义的。

简单来说，我们将每个离散的像素值看作一个小箱子（Bin）。对于缩放后的像素值 $x_0^i$，它的概率不是 $p(x_0^i)$，而是高斯分布在 $[x_0^i - \frac{1}{255}, x_0^i + \frac{1}{255}]$ 这个区间内的积分。

我们不详细叙述这些内容。推荐你读 https://arxiv.org/abs/2006.11239 Denoising Diffusion Probabilistic Models 了解原文。

### DDPM 的问题与解决

DDPM 最大的问题就是单次推理是单步的，这意味着从 $T = 1000$ 开始减噪还原就需要推理 1000 次，对于大型神经网络来说很难接受。本章篇幅不太足够了。在下一章，我为你介绍 Denoising Diffusion Implicit Models，我们通过简单的数学事实，转换训练的思路，发现推理实际上可以跳步。

# 总结

不过话说回来，如果我们强行让 DDPM 一次推理多步，会如何？实际上大概率会给出模糊的影像甚至无意义内容。原因是在 DDPM 的假设中，也就是我们所说的极小步长下逆向去噪近似高斯分布，在大步长下难以成立。我们还是快速进入下一章吧。